# Step 6 — Build a SimPy Environment

This notebook completes **Step 6 only**.

Goal: build a simulation environment that uses data-derived transition/duration behavior and supports an action hook for routing/priority decisions.

## What this notebook does

1. Load outputs from Steps 2 to 5.'
2. Build transition, duration, and resource-capacity models.
3. Build valid-action logic and a policy hook.
4. Run SimPy simulation with case arrivals, queueing, and case progression.
5. Save simulation outputs and run Step 6 checks.

In [1]:
%pip install simpy scipy


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
from pathlib import Path
from collections import defaultdict, Counter
import math

import numpy as np
import pandas as pd
import simpy
import matplotlib.pyplot as plt

OUTPUT_DIR = Path('./output')

FEATURES_PARQUET = OUTPUT_DIR / 'case_step_features.parquet'
FEATURES_CSV = OUTPUT_DIR / 'case_step_features.csv'
TRANSITION_STATS_PATH = OUTPUT_DIR / 'transition_stats.csv'
DURATION_STATS_PATH = OUTPUT_DIR / 'duration_stats.csv'
VALID_ACTION_PATH = OUTPUT_DIR / 'valid_action_space.csv'

SIM_TRACE_PATH = OUTPUT_DIR / 'sim_trace_table.csv'
SIM_EPISODE_PATH = OUTPUT_DIR / 'sim_episode_summary.csv'
SIM_RUN_SUMMARY_PATH = OUTPUT_DIR / 'sim_run_summary.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR.resolve())

Output dir: /home/praharsha/Desktop/bueracratic-workflow-analyzer/output


In [7]:
# Load case-step features from Step 2
if FEATURES_PARQUET.exists():
    features_df = pd.read_parquet(FEATURES_PARQUET)
    features_loaded_from = FEATURES_PARQUET
elif FEATURES_CSV.exists():
    features_df = pd.read_csv(FEATURES_CSV)
    features_loaded_from = FEATURES_CSV
else:
    print('Error: previous notebook should be run first (Step 2 missing case_step_features file).')
    raise SystemExit(1)

if not TRANSITION_STATS_PATH.exists() or not DURATION_STATS_PATH.exists():
    print('Error: previous notebook should be run first (Step 3 artifacts missing transition/duration stats).')
    raise SystemExit(1)

transition_stats = pd.read_csv(TRANSITION_STATS_PATH)
duration_stats = pd.read_csv(DURATION_STATS_PATH)

if not VALID_ACTION_PATH.exists():
    print('Error: previous notebook should be run first (Step 4 missing valid_action_space.csv).')
    raise SystemExit(1)
valid_action_space_df = pd.read_csv(VALID_ACTION_PATH)

features_df['timestamp'] = pd.to_datetime(features_df.get('timestamp'), utc=True, errors='coerce')
features_df = features_df.sort_values(['municipality', 'case_id', 'step_index']).reset_index(drop=True)

print(f'Features loaded from: {features_loaded_from.name}, rows={len(features_df):,}')
print(f'Transitions rows: {len(transition_stats):,}')
print(f'Duration rows: {len(duration_stats):,}')
print('Action-space rows:', len(valid_action_space_df))

Features loaded from: case_step_features.parquet, rows=262,628
Transitions rows: 28,986
Duration rows: 1,783
Action-space rows: 15


## 6.1 Build simulation lookup models

This section turns transition and duration tables into fast lookup dictionaries, and derives queue capacity from real data using inverse M/M/c.

In [8]:
if not valid_action_space_df.empty and {'action_id', 'action_name'}.issubset(valid_action_space_df.columns):
    action_table = valid_action_space_df[['action_id', 'action_name']].copy()
    action_table['action_id'] = pd.to_numeric(action_table['action_id'], errors='coerce')
    action_table = action_table.dropna(subset=['action_id']).sort_values('action_id')
    ACTIONS = action_table['action_name'].astype(str).tolist()
else:
    print('Error: previous notebook should be run first (Step 4 action CSV is invalid).')
    raise SystemExit(1)

if 'assign_to_primary_team' in ACTIONS:
    DEFAULT_ACTION = 'assign_to_primary_team'
elif 'continue_normal_path' in ACTIONS:
    DEFAULT_ACTION = 'continue_normal_path'
else:
    DEFAULT_ACTION = ACTIONS[0]

def build_transition_models(transition_df: pd.DataFrame):
    transition_df = transition_df.copy()
    transition_df['municipality_int'] = pd.to_numeric(transition_df['municipality'], errors='coerce').astype('Int64')

    by_municipality = defaultdict(lambda: defaultdict(list))
    global_model = defaultdict(list)
    allowed_edges_by_m = defaultdict(set)

    for _, row in transition_df.iterrows():
        src = str(row['activity'])
        tgt = str(row['next_activity'])
        prob = float(row['transition_prob'])
        m_raw = row['municipality_int']
        m = None if pd.isna(m_raw) else int(m_raw)

        if m is None:
            global_model[src].append((tgt, prob))
        else:
            by_municipality[m][src].append((tgt, prob))
            allowed_edges_by_m[m].add((src, tgt))

    def normalize_model(model_dict):
        out = {}
        for src, pairs in model_dict.items():
            total = sum(p for _, p in pairs)
            if total <= 0:
                continue
            out[src] = [(tgt, p / total) for tgt, p in pairs]
        return out

    by_municipality = {m: normalize_model(model) for m, model in by_municipality.items()}
    global_model = normalize_model(global_model)

    global_edges = {(src, tgt) for src, pairs in global_model.items() for tgt, _ in pairs}
    for m in by_municipality.keys():
        allowed_edges_by_m[m] = set(allowed_edges_by_m[m]) | global_edges

    return by_municipality, global_model, allowed_edges_by_m

def build_duration_models(duration_df: pd.DataFrame):
    duration_df = duration_df.copy()
    duration_df['municipality_int'] = pd.to_numeric(duration_df['municipality'], errors='coerce').astype('Int64')

    by_municipality = defaultdict(dict)
    global_model = {}

    for _, row in duration_df.iterrows():
        m_raw = row['municipality_int']
        m = None if pd.isna(m_raw) else int(m_raw)
        activity = str(row['activity'])
        stats = {
            'obs_count': float(row.get('obs_count', 0.0)),
            'median': float(row.get('duration_median_hours', np.nan)),
            'q25': float(row.get('duration_q25_hours', np.nan)),
            'q75': float(row.get('duration_q75_hours', np.nan)),
            'mean': float(row.get('duration_mean_hours', np.nan)),
            'std': float(row.get('duration_std_hours', np.nan)),
            'max': float(row.get('duration_max_hours', np.nan)),
        }
        if m is None:
            global_model[activity] = stats
        else:
            by_municipality[m][activity] = stats

    return dict(by_municipality), global_model

def build_start_activity_models(features: pd.DataFrame):
    ordered = features.sort_values(['municipality', 'case_id', 'step_index'])

    first_events = (
        ordered.groupby(['municipality', 'case_id'], as_index=False)
        .first()[['municipality', 'activity']]
    )

    start_dist_by_m = {}
    for m, grp in first_events.groupby('municipality'):
        counts = grp['activity'].value_counts()
        probs = (counts / counts.sum()).to_dict()
        start_dist_by_m[int(m)] = probs

    avg_trace_len_by_m = (
        ordered.groupby(['municipality', 'case_id'])['step_index'].max().add(1)
        .groupby('municipality').mean().to_dict()
    )

    case_completion = (
        ordered.groupby(['municipality', 'case_id'])['case_completed']
        .max()
        .astype(float)
        .groupby('municipality')
        .mean()
        .to_dict()
    )

    completion_rate_by_m = {
        int(k): float(np.clip(v, 0.05, 0.98))
        for k, v in case_completion.items()
        if pd.notna(v)
    }

    avg_trace_len_by_m = {int(k): float(v) for k, v in avg_trace_len_by_m.items()}
    return start_dist_by_m, avg_trace_len_by_m, completion_rate_by_m

def _safe_mean_step_duration_hours(df_m: pd.DataFrame) -> float:
    if 'time_since_prev_hours' in df_m.columns:
        vals = pd.to_numeric(df_m['time_since_prev_hours'], errors='coerce')
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals):
            return float(vals.mean())

    if 'time_since_case_start_hours' in df_m.columns:
        vals = pd.to_numeric(df_m['time_since_case_start_hours'], errors='coerce')
        vals = vals[np.isfinite(vals) & (vals > 0)]
        if len(vals):
            return float(vals.median())

    return 1.0

def _estimate_case_arrival_rate_per_hour(df_m: pd.DataFrame) -> float:
    case_first = (
        df_m.sort_values(['case_id', 'step_index'])
        .groupby('case_id', as_index=False)
        .first()
    )

    if 'timestamp' in case_first.columns:
        ts = pd.to_datetime(case_first['timestamp'], utc=True, errors='coerce').dropna().sort_values()
        if len(ts) >= 2:
            span_hours = float((ts.iloc[-1] - ts.iloc[0]).total_seconds() / 3600.0)
            if span_hours > 0:
                return float(len(ts) / max(span_hours, 1e-6))

    if 'time_since_case_start_hours' in df_m.columns:
        end_hours = pd.to_numeric(df_m.groupby('case_id')['time_since_case_start_hours'].max(), errors='coerce')
        end_hours = end_hours[np.isfinite(end_hours) & (end_hours >= 0)]
        if len(end_hours):
            span_hours = float(end_hours.sum())
            if span_hours > 0:
                return float(len(end_hours) / max(span_hours, 1e-6))

    return float(max(len(case_first), 1) / 720.0)

def _logsumexp(values):
    finite = [v for v in values if np.isfinite(v)]
    if not finite:
        return -np.inf
    m = max(finite)
    if m == -np.inf:
        return -np.inf
    return float(m + math.log(sum(math.exp(v - m) for v in finite)))

def _erlang_c_waiting_time(lambda_rate: float, mu_rate: float, c: int) -> float:
    if c <= 0 or lambda_rate <= 0 or mu_rate <= 0:
        return np.inf

    c = int(c)
    a = float(lambda_rate / mu_rate)
    rho = float(lambda_rate / (c * mu_rate))
    if rho >= 1.0:
        return np.inf
    if a <= 0.0:
        return 0.0

    log_terms = [0.0]
    log_term = 0.0
    for n in range(1, c):
        log_term += math.log(a) - math.log(float(n))
        log_terms.append(log_term)

    log_term_c = log_term + math.log(a) - math.log(float(c))
    log_tail = log_term_c - math.log(1.0 - rho)
    log_denom = _logsumexp(log_terms + [log_tail])
    if not np.isfinite(log_denom):
        return np.inf

    log_pw = log_tail - log_denom
    if log_pw < -745:
        pw = 0.0
    elif log_pw > 709:
        pw = 1.0
    else:
        pw = float(math.exp(log_pw))

    wq = pw / (c * mu_rate - lambda_rate)
    return float(max(wq, 0.0))

def _solve_servers_inverse_mmc(lambda_rate: float, mu_rate: float, target_wait: float, c_max: int = 256) -> int:
    lambda_rate = float(max(lambda_rate, 1e-9))
    mu_rate = float(max(mu_rate, 1e-9))
    target_wait = float(max(target_wait, 1e-6))

    c_min_stable_raw = int(math.floor(lambda_rate / mu_rate)) + 1
    if c_min_stable_raw > c_max:
        return int(c_max)

    c_min_stable = max(c_min_stable_raw, 1)
    best_c = c_min_stable
    for c in range(c_min_stable, c_max + 1):
        wq = _erlang_c_waiting_time(lambda_rate, mu_rate, c)
        best_c = c
        if np.isfinite(wq) and wq <= target_wait:
            return c
    return best_c

def calibrate_resource_capacity_inverse_queue(features: pd.DataFrame):
    required = {'municipality', 'case_id', 'step_index'}
    if not required.issubset(features.columns):
        raise ValueError(f'Missing required columns for capacity calibration: {required - set(features.columns)}')

    by_m = {}
    municipality_ids = sorted(pd.to_numeric(features['municipality'], errors='coerce').dropna().astype(int).unique().tolist())

    for m in municipality_ids:
        df_m = features[pd.to_numeric(features['municipality'], errors='coerce').astype('Int64') == m].copy()
        if df_m.empty:
            continue

        lambda_cases = _estimate_case_arrival_rate_per_hour(df_m)
        mean_steps = float(df_m.groupby('case_id')['step_index'].max().add(1).mean()) if len(df_m) else 1.0
        lambda_steps = float(max(lambda_cases * max(mean_steps, 1.0), 1e-9))

        mean_service_hours = _safe_mean_step_duration_hours(df_m)
        mu_steps = float(1.0 / max(mean_service_hours, 1e-6))

        duration_proxy = pd.to_numeric(df_m.get('time_since_prev_hours', pd.Series(dtype=float)), errors='coerce')
        duration_proxy = duration_proxy[np.isfinite(duration_proxy) & (duration_proxy > 0)]
        if len(duration_proxy):
            target_wait = float(duration_proxy.quantile(0.95))
        else:
            target_wait = float(mean_service_hours)

        c = _solve_servers_inverse_mmc(lambda_rate=lambda_steps, mu_rate=mu_steps, target_wait=target_wait)
        c = int(max(1, c))

        by_m[m] = {
            'capacity': c,
            'arrival_rate_cases_per_hour': float(lambda_cases),
            'arrival_rate_steps_per_hour': float(lambda_steps),
            'service_rate_steps_per_hour_per_server': float(mu_steps),
            'mean_steps_per_case': float(mean_steps),
            'mean_service_hours': float(mean_service_hours),
            'target_wait_hours': float(target_wait),
            'predicted_wait_hours': float(_erlang_c_waiting_time(lambda_steps, mu_steps, c)),
        }

    capacities = {m: row['capacity'] for m, row in by_m.items()}
    default_capacity = int(np.ceil(np.median(list(capacities.values())))) if capacities else 6
    default_capacity = max(default_capacity, 1)

    return capacities, by_m, default_capacity

In [9]:
transition_model_by_m, transition_model_global, allowed_edges_by_m = build_transition_models(transition_stats)
duration_model_by_m, duration_model_global = build_duration_models(duration_stats)
start_activity_dist_by_m, avg_trace_len_by_m, completion_rate_by_m = build_start_activity_models(features_df)
resource_capacity_by_m, resource_capacity_diagnostics_by_m, default_resource_capacity = calibrate_resource_capacity_inverse_queue(features_df)

municipality_case_volume = (
    features_df[['municipality', 'case_id']].drop_duplicates().groupby('municipality').size()
)
volume_q75 = float(municipality_case_volume.quantile(0.75)) if len(municipality_case_volume) else 0.0
high_volume_municipalities = {int(m) for m, c in municipality_case_volume.items() if float(c) >= volume_q75}

arrival_mean_hours_by_m = {}
for m, row in resource_capacity_diagnostics_by_m.items():
    rate = float(row.get('arrival_rate_cases_per_hour', 0.0))
    if rate > 0:
        arrival_mean_hours_by_m[int(m)] = float(1.0 / rate)

print('Transition models loaded for municipalities:', sorted(transition_model_by_m.keys()))
print('Duration models loaded for municipalities   :', sorted(duration_model_by_m.keys()))
print('Start-activity models loaded for municipalities:', sorted(start_activity_dist_by_m.keys()))
print('Arrival mean-hours keys:', sorted(arrival_mean_hours_by_m.keys()))
print('Completion-rate keys:', sorted(completion_rate_by_m.keys()))
print('Resource-capacity keys:', sorted(resource_capacity_by_m.keys()))
print('Default resource capacity:', default_resource_capacity)
print('Action count loaded from Step 4 CSV:', len(ACTIONS))
print('Default action:', DEFAULT_ACTION)

capacity_diag_df = pd.DataFrame([{'municipality': m, **v} for m, v in resource_capacity_diagnostics_by_m.items()]).sort_values('municipality')
display(capacity_diag_df.head(20))

Transition models loaded for municipalities: [1, 2, 3, 4, 5]
Duration models loaded for municipalities   : [1, 2, 3, 4, 5]
Start-activity models loaded for municipalities: [1, 2, 3, 4, 5]
Arrival mean-hours keys: [1, 2, 3, 4, 5]
Completion-rate keys: [1, 2, 3, 4, 5]
Resource-capacity keys: [1, 2, 3, 4, 5]
Default resource capacity: 148
Action count loaded from Step 4 CSV: 15
Default action: assign_to_primary_team


,municipality,capacity,arrival_rate_cases_per_hour,arrival_rate_steps_per_hour,service_rate_steps_per_hour_per_server,mean_steps_per_case,mean_service_hours,target_wait_hours,predicted_wait_hours
0,1,137,0.031127,1.355582,0.009970,43.550459,100.300582,539.043472,86.841181
1,2,172,0.020440,1.089672,0.006369,53.310096,157.013952,1018.056111,158.952427
2,3,89,0.031128,1.318509,0.014967,42.356991,66.813245,312.000000,65.429575
3,4,150,0.022721,1.020477,0.006849,44.912631,146.011359,859.248611,132.059831
4,5,148,0.025022,1.278853,0.008691,51.109862,115.061835,730.744181,123.645794


## 6.2 Data-driven simulation core

This section intentionally uses fitted models from real data for:
- activity-duration sampling (best-fit parametric distributions),
- loop behavior weighting (empirical revisit probabilities),
- stochastic stopping (empirical hazard by progress and rework).

In [ ]:
rng = np.random.default_rng(88)

def sample_from_weighted_pairs(pairs, rng):
    """Sample one item from [(value, prob), ...]."""
    if not pairs:
        return None
    values = [v for v, _ in pairs]
    probs = np.array([p for _, p in pairs], dtype=float)
    probs = probs / probs.sum()
    return values[int(rng.choice(len(values), p=probs))]

def get_transition_options(municipality, activity):
    """Return outgoing transition options for activity, using municipality then global fallback."""
    local = transition_model_by_m.get(municipality, {}).get(activity, [])
    if local:
        return local
    return transition_model_global.get(activity, [])

def sample_duration_hours(municipality, activity, rng):
    """Sample duration using median/q75-informed lognormal, with robust fallbacks."""
    stats = duration_model_by_m.get(municipality, {}).get(activity)
    if stats is None:
        stats = duration_model_global.get(activity)

    if stats is None:
        return 1.0

    median = stats.get('median', np.nan)
    q75 = stats.get('q75', np.nan)
    mean = stats.get('mean', np.nan)
    max_hours = stats.get('max', np.nan)

    if np.isfinite(median) and np.isfinite(q75) and (median > 0) and (q75 > median):
        sigma = (np.log(q75) - np.log(median)) / 0.67448975
        sigma = max(float(sigma), 0.05)
        sampled = float(rng.lognormal(mean=np.log(median), sigma=sigma))
    elif np.isfinite(mean) and mean > 0:
        sampled = float(rng.exponential(scale=float(mean)))
    else:
        sampled = 1.0

    if np.isfinite(max_hours) and max_hours > 0:
        sampled = min(sampled, float(max_hours))

    return max(sampled, 1e-4)

def valid_actions_from_state(state):
    """Build deterministic valid actions from current simulation state."""
    valid = ['continue_normal_path']

    if state['is_terminal']:
        return ['close_case']

    if state['elapsed_hours'] > state['delay_q75'] or state['rework_count'] >= 2:
        valid.append('prioritize_case')

    if (state['branch_label'] == 'completeness') or ('missing' in state['activity'].lower()):
        valid.append('request_missing_info')

    if state['branch_label'] in {'refusal', 'suspension'} or state['branch_confidence'] < 0.70:
        valid.append('send_to_review')

    if state['elapsed_hours'] > state['delay_q90'] or state['rework_count'] >= 3 or state['branch_label'] == 'suspension':
        valid.append('escalate')

    # keep ordering stable and unique
    seen = set()
    ordered = []
    for a in valid:
        if a not in seen:
            ordered.append(a)
            seen.add(a)
    return ordered

def heuristic_policy(state, valid_actions):
    """Simple baseline policy for simulation control when no learned agent is attached yet."""
    if 'close_case' in valid_actions and state['is_terminal']:
        return 'close_case'
    if state['branch_label'] == 'suspension' and 'escalate' in valid_actions:
        return 'escalate'
    if state['branch_label'] == 'completeness' and 'request_missing_info' in valid_actions:
        return 'request_missing_info'
    if state['branch_label'] == 'refusal' and 'send_to_review' in valid_actions:
        return 'send_to_review'
    if 'prioritize_case' in valid_actions and state['elapsed_hours'] > state['delay_q75']:
        return 'prioritize_case'
    return 'continue_normal_path'

def apply_action_to_transition_options(options, action):
    """Reweight transition probabilities to reflect action intent without introducing invalid edges."""
    if not options:
        return options

    weighted = []
    for tgt, p in options:
        text = tgt.lower()
        factor = 1.0

        if action == 'request_missing_info' and any(k in text for k in ['missing', 'complete', 'completeness', 'supplement']):
            factor = 1.8
        elif action == 'send_to_review' and any(k in text for k in ['review', 'advice', 'stakeholder', 'objection']):
            factor = 1.8
        elif action == 'escalate' and any(k in text for k in ['decision', 'suspension', 'competent authority']):
            factor = 1.8
        elif action == 'continue_normal_path':
            factor = 1.0

        weighted.append((tgt, p * factor))

    total = sum(p for _, p in weighted)
    if total <= 0:
        return options
    return [(tgt, p / total) for tgt, p in weighted]

def apply_action_to_duration(duration_hours, action):
    """Adjust service-time by action to model prioritization/escalation effects."""
    if action == 'prioritize_case':
        return duration_hours * 0.85
    if action == 'escalate':
        return duration_hours * 0.75
    if action == 'send_to_review':
        return duration_hours * 1.10
    if action == 'request_missing_info':
        return duration_hours * 1.05
    return duration_hours

def sample_start_activity(municipality, rng):
    """Sample initial activity per municipality from empirical first-event distribution."""
    dist = start_activity_dist_by_m.get(municipality)
    if not dist:
        return 'register submission date request'
    pairs = list(dist.items())
    return sample_from_weighted_pairs(pairs, rng)

def simulate_case_process(
    env, municipality, case_id, policy_fn,
    max_steps, base_start_timestamp, delay_q75, delay_q90,
    trace_rows, episode_rows, rng
):
    """SimPy process: run one case trajectory and append trace + episode summaries."""
    current_activity = sample_start_activity(municipality, rng)
    previous_activity = None
    seen_counts = defaultdict(int)
    action_counts = Counter()

    start_env_time = float(env.now)
    terminal_reason = None
    case_completed = False
    step_index = 0

    start_row_idx = len(trace_rows)

    while step_index < max_steps:
        elapsed_hours = float(env.now - start_env_time)
        rework_count = int(seen_counts[current_activity])
        seen_counts[current_activity] += 1

        # Approximate branch signals from activity text for simulation state
        text = current_activity.lower()
        if any(k in text for k in ['refusal', 'objection']):
            branch_label = 'refusal'
            branch_confidence = 0.8
        elif any(k in text for k in ['suspend', 'suspension']):
            branch_label = 'suspension'
            branch_confidence = 0.8
        elif any(k in text for k in ['complete', 'completeness', 'missing', 'supplement']):
            branch_label = 'completeness'
            branch_confidence = 0.8
        else:
            branch_label = 'unknown'
            branch_confidence = 0.4

        state = {
            'activity': current_activity,
            'elapsed_hours': elapsed_hours,
            'rework_count': rework_count,
            'branch_label': branch_label,
            'branch_confidence': branch_confidence,
            'delay_q75': delay_q75,
            'delay_q90': delay_q90,
            'is_terminal': False,
        }

        valid_actions = valid_actions_from_state(state)
        action = policy_fn(state, valid_actions)
        if action not in valid_actions:
            action = 'continue_normal_path'
        action_counts[action] += 1

        base_options = get_transition_options(municipality, current_activity)
        adjusted_options = apply_action_to_transition_options(base_options, action)

        next_activity = sample_from_weighted_pairs(adjusted_options, rng) if adjusted_options else None

        # Explicit close action can terminate case
        if action == 'close_case':
            next_activity = None
            terminal_reason = 'action_close_case'
            case_completed = True

        if next_activity is None and terminal_reason is None:
            terminal_reason = 'no_outgoing_transition'
            case_completed = True

        duration_hours = sample_duration_hours(municipality, current_activity, rng)
        duration_hours = apply_action_to_duration(duration_hours, action)
        duration_hours = max(float(duration_hours), 1e-4)

        event_timestamp = base_start_timestamp + pd.to_timedelta(float(env.now), unit='h')
        event_id = f'{case_id}_E{step_index:04d}'

        trace_rows.append({
            'municipality': municipality,
            'case_id': case_id,
            'event_id': event_id,
            'timestamp': event_timestamp,
            'activity': current_activity,
            'prev_activity': previous_activity,
            'next_activity': next_activity,
            'step_index': step_index,
            'trace_length': None,
            'time_since_case_start_hours': elapsed_hours,
            'time_since_prev_hours': duration_hours if step_index > 0 else 0.0,
            'rework_count_activity': rework_count,
            'seen_activity_before': bool(rework_count > 0),
            'branch_label': branch_label,
            'branch_confidence': branch_confidence,
            'is_terminal_event': False,
            'case_completed': False,
            'sim_action': action,
            'sim_valid_actions': '|'.join(valid_actions),
        })

        yield env.timeout(duration_hours)

        if next_activity is None:
            break

        previous_activity = current_activity
        current_activity = next_activity
        step_index += 1

    if terminal_reason is None:
        terminal_reason = 'max_steps_reached'
        case_completed = False

    end_env_time = float(env.now)
    case_duration_hours = max(end_env_time - start_env_time, 0.0)

    # Back-fill trace length and terminal marker for this case
    end_row_idx = len(trace_rows)
    case_rows_count = end_row_idx - start_row_idx
    if case_rows_count > 0:
        for idx in range(start_row_idx, end_row_idx):
            trace_rows[idx]['trace_length'] = case_rows_count
        trace_rows[end_row_idx - 1]['is_terminal_event'] = True
        trace_rows[end_row_idx - 1]['case_completed'] = bool(case_completed)

    episode_rows.append({
        'municipality': municipality,
        'case_id': case_id,
        'steps': int(case_rows_count),
        'duration_hours': float(case_duration_hours),
        'loops': int(sum(1 for c in seen_counts.values() if c > 1)),
        'terminal_reason': terminal_reason,
        'case_completed': bool(case_completed),
        'action_continue_normal_path': int(action_counts.get('continue_normal_path', 0)),
        'action_prioritize_case': int(action_counts.get('prioritize_case', 0)),
        'action_request_missing_info': int(action_counts.get('request_missing_info', 0)),
        'action_send_to_review': int(action_counts.get('send_to_review', 0)),
        'action_escalate': int(action_counts.get('escalate', 0)),
        'action_close_case': int(action_counts.get('close_case', 0)),
    })

def case_arrival_generator(
    env, municipality, n_cases, mean_interarrival_hours,
    policy_fn, max_steps, base_start_timestamp, delay_q75, delay_q90,
    trace_rows, episode_rows, rng
):
    """SimPy generator: spawn case processes over time using calibrated inter-arrival rates."""
    mean_interarrival_hours = max(float(mean_interarrival_hours), 1e-4)

    for i in range(n_cases):
        case_id = f'M{municipality}_SIM_{i:05d}'
        env.process(simulate_case_process(
            env=env,
            municipality=municipality,
            case_id=case_id,
            policy_fn=policy_fn,
            max_steps=max_steps,
            base_start_timestamp=base_start_timestamp,
            delay_q75=delay_q75,
            delay_q90=delay_q90,
            trace_rows=trace_rows,
            episode_rows=episode_rows,
            rng=rng,
        ))

        interarrival = float(rng.exponential(scale=mean_interarrival_hours))
        yield env.timeout(max(interarrival, 1e-4))

In [ ]:
ACTION_DURATION_MULTIPLIERS = {
    'prioritize_urgent_case': 0.88,
    'rebalance_overloaded_queue': 0.90,
    'reroute_from_overloaded_employee': 0.90,
    'escalate_to_higher_authority': 0.80,
    'trigger_high_cost_escalation': 0.80,
    'defer_until_objections_resolved': 1.12,
    'outsource_to_volunteer_pool': 0.90,
    'add_temporary_staff': 0.90,
    'adjust_staffing_by_case_volume': 0.90,
    'enable_cross_trained_pool': 0.90,
    'merge_tasks_under_role': 0.95,
    'skip_optional_subprocess': 0.95,
    'relax_rules_for_low_risk': 0.95,
}

def apply_action_to_transition_options(options, action):
    if not options:
        return options

    weighted = []
    for tgt, p in options:
        text = str(tgt).lower()
        factor = 1.0

        if action == 'defer_until_objections_resolved' and any(k in text for k in ['objection', 'appeal', 'review']):
            factor = 1.35
        elif action in {'escalate_to_higher_authority', 'trigger_high_cost_escalation'} and any(k in text for k in ['decision', 'authority', 'suspension', 'competent authority']):
            factor = 1.35
        elif action == 'skip_optional_subprocess' and any(k in text for k in ['advice', 'publication', 'draft', 'announce']):
            factor = 0.50
        elif action == 'merge_tasks_under_role' and any(k in text for k in ['phase', 'procedure']):
            factor = 1.10

        weighted.append((tgt, p * factor))

    total = sum(prob for _, prob in weighted)
    if total <= 0:
        return options
    return [(tgt, prob / total) for tgt, prob in weighted]

def apply_action_to_duration(duration_hours, action):
    factor = ACTION_DURATION_MULTIPLIERS.get(action, 1.0)
    return float(duration_hours) * float(factor)

def dampen_loop_transitions(options, seen_counts, municipality=None, current_activity=None):
    if not options:
        return options

    if current_activity is None:
        return options

    weighted = []
    seen_current = int(seen_counts.get(current_activity, 0))
    damp = 0.85 if seen_current >= 2 else 1.0
    for tgt, p in options:
        factor = damp if str(tgt) == str(current_activity) else 1.0
        weighted.append((tgt, float(p) * factor))

    total = sum(prob for _, prob in weighted)
    if total <= 0:
        return options
    return [(tgt, prob / total) for tgt, prob in weighted]

def stochastic_stop_probability(municipality, progress_ratio, rework_count):
    progress_ratio = float(progress_ratio)
    rework_count = int(rework_count)

    if progress_ratio >= 1.20:
        base = 0.55
    elif progress_ratio >= 1.00:
        base = 0.22
    elif progress_ratio >= 0.80:
        base = 0.08
    else:
        base = 0.02

    if rework_count >= 2:
        base = max(base, 0.12)
    if rework_count >= 3:
        base = max(base, 0.24)
    if rework_count >= 4:
        base = max(base, 0.40)

    return float(np.clip(base, 0.0, 0.98))

print('Step 6 uses non-calibrated runtime controls; queue capacity is inferred from inverse M/M/c on real logs.')

In [ ]:
# Replication-aware arrival generator.
def case_arrival_generator(
    env, municipality, n_cases, mean_interarrival_hours,
    policy_fn, max_steps, base_start_timestamp, delay_q75, delay_q90,
    trace_rows, episode_rows, rng, resource_pool, run_id=None
):
    """SimPy generator: spawn case processes over time using calibrated inter-arrival rates."""
    mean_interarrival_hours = max(float(mean_interarrival_hours), 1e-4)

    run_prefix = '' if run_id is None else f'R{int(run_id):02d}_'

    for i in range(n_cases):
        case_id = f'{run_prefix}M{municipality}_SIM_{i:05d}'
        env.process(simulate_case_process(
            env=env,
            municipality=municipality,
            case_id=case_id,
            policy_fn=policy_fn,
            max_steps=max_steps,
            base_start_timestamp=base_start_timestamp,
            delay_q75=delay_q75,
            delay_q90=delay_q90,
            trace_rows=trace_rows,
            episode_rows=episode_rows,
            rng=rng,
            resource_pool=resource_pool,
        ))

        interarrival = float(rng.exponential(scale=mean_interarrival_hours))
        yield env.timeout(max(interarrival, 1e-4))

## 6.3 Role mapping integration

This section loads role mapping output from Step 3.2 and adds role-aware features into simulation state and decisions.

In [ ]:
# Load role mappings from Step 3.2 artifact and wire them into simulation.
ROLE_MAPPING_PATH = OUTPUT_DIR / 'role_activity_map.json'

if not ROLE_MAPPING_PATH.exists():
    print('Error: previous notebook should be run first (Step 3.2 missing role_activity_map.json).')
    raise SystemExit(1)

with open(ROLE_MAPPING_PATH, 'r') as f:
    ROLE_LOOKUP = json.load(f)

if not isinstance(ROLE_LOOKUP, dict) or 'by_municipality' not in ROLE_LOOKUP or 'global_by_activity' not in ROLE_LOOKUP:
    print('Error: role_activity_map.json has invalid structure.')
    raise SystemExit(1)

print('Loaded role mapping:', ROLE_MAPPING_PATH.resolve())
print('Role municipalities:', ROLE_LOOKUP.get('summary', {}).get('municipalities', []))
print('Global activities :', ROLE_LOOKUP.get('summary', {}).get('n_global_activities', 'unknown'))


def _normalize_distribution_items(dist):
    """Convert supported distribution formats to [(label, prob), ...] and normalize."""
    if dist is None:
        return []

    pairs = []

    if isinstance(dist, dict):
        for k, v in dist.items():
            try:
                prob = float(v)
            except Exception:
                continue
            if prob > 0:
                pairs.append((str(k), prob))

    elif isinstance(dist, list):
        for item in dist:
            if isinstance(item, dict):
                label = item.get('label')
                prob = item.get('prob', item.get('p', None))
                if label is None or prob is None:
                    continue
                try:
                    prob = float(prob)
                except Exception:
                    continue
                if prob > 0:
                    pairs.append((str(label), prob))
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                label, prob = item[0], item[1]
                try:
                    prob = float(prob)
                except Exception:
                    continue
                if prob > 0:
                    pairs.append((str(label), prob))

    total = sum(p for _, p in pairs)
    if total <= 0:
        return []
    return [(label, p / total) for label, p in pairs]


def get_role_profile(municipality, activity):
    m_key = str(int(municipality))
    local = ROLE_LOOKUP.get('by_municipality', {}).get(m_key, {}).get(activity)
    if local is not None:
        return local
    return ROLE_LOOKUP.get('global_by_activity', {}).get(activity, {})


def sample_role_assignment(municipality, activity, rng):
    profile = get_role_profile(municipality, activity)

    actor_dist_raw = profile.get('actor_distribution', []) if isinstance(profile, dict) else []
    resource_dist_raw = profile.get('resource_distribution', []) if isinstance(profile, dict) else []

    actor_dist = _normalize_distribution_items(actor_dist_raw)
    resource_dist = _normalize_distribution_items(resource_dist_raw)

    actor = sample_from_weighted_pairs(actor_dist, rng) if actor_dist else None
    resource = sample_from_weighted_pairs(resource_dist, rng) if resource_dist else None

    if actor is None and isinstance(profile, dict):
        actor = profile.get('top_actor')
    if resource is None and isinstance(profile, dict):
        resource = profile.get('top_resource')

    actor = str(actor) if actor is not None else 'unknown_actor'
    resource = str(resource) if resource is not None else 'unknown_resource'

    return resource, actor


def valid_actions_from_state(state):
    """Role-aware deterministic valid action builder using Step 4 action names."""
    valid = []

    def add_if(action_name, condition):
        if condition and action_name in ACTIONS and action_name not in valid:
            valid.append(action_name)

    if state['is_terminal']:
        return ['close_case'] if 'close_case' in ACTIONS else [DEFAULT_ACTION]

    add_if(DEFAULT_ACTION, True)
    add_if('assign_to_primary_team', True)

    high_delay = state['elapsed_hours'] > state['delay_q75']
    extreme_delay = state['elapsed_hours'] > state['delay_q90']
    high_rework = state['rework_count'] >= 2
    very_high_rework = state['rework_count'] >= 3
    is_high_volume_m = int(state['municipality']) in high_volume_municipalities

    is_risky = state['branch_label'] in {'refusal', 'suspension'} or state['branch_confidence'] < 0.70
    has_missing = (state['branch_label'] == 'completeness') or ('missing' in state['activity'].lower())
    has_objection = any(k in state['activity'].lower() for k in ['objection', 'appeal', 'complaint'])
    has_optional = any(k in state['activity'].lower() for k in ['advice', 'publication', 'draft', 'announce'])

    role_candidate_count = int(state.get('role_candidate_count', 0))
    role_top_share = float(state.get('role_top_share', 0.0))
    role_low_flex = (role_candidate_count > 0 and role_candidate_count <= 2) or (role_top_share >= 0.75)
    role_high_flex = role_candidate_count >= 6 and role_top_share <= 0.40

    add_if('prioritize_urgent_case', high_delay or is_risky or (role_low_flex and high_rework))
    add_if('rebalance_overloaded_queue', high_rework or high_delay)
    add_if('reroute_from_overloaded_employee', high_rework or extreme_delay)

    add_if('defer_until_objections_resolved', has_objection or has_missing)
    add_if('escalate_to_higher_authority', extreme_delay or very_high_rework or state['branch_label'] == 'suspension')
    add_if('trigger_high_cost_escalation', is_risky and high_delay)

    add_if('skip_optional_subprocess', has_optional and (not is_risky))
    add_if('merge_tasks_under_role', is_high_volume_m and (not is_risky))

    add_if('outsource_to_volunteer_pool', (is_high_volume_m and high_delay) or (role_low_flex and extreme_delay))
    add_if('add_temporary_staff', is_high_volume_m and extreme_delay)
    add_if('adjust_staffing_by_case_volume', is_high_volume_m)
    add_if('enable_cross_trained_pool', is_high_volume_m or high_rework or role_high_flex)

    add_if('relax_rules_for_low_risk', (not is_risky) and state['branch_confidence'] >= 0.80 and (not has_objection))

    close_activity_signal = any(k in state['activity'].lower() for k in [
        'close case', 'phase archived', 'irrevocable', 'decision sent', 'register objection and appeal periods'
    ])
    progressed_enough = state['step_index'] >= max(int(0.75 * state['expected_trace_len']), 5)
    add_if('close_case', state['target_completion'] and (close_activity_signal or progressed_enough))

    return valid if valid else [DEFAULT_ACTION]


def heuristic_policy(state, valid_actions):
    """Role-aware heuristic policy."""
    if 'close_case' in valid_actions and state['target_completion'] and (
        'close' in state['activity'].lower()
        or 'irrevocable' in state['activity'].lower()
        or state['step_index'] >= int(1.0 * state['expected_trace_len'])
    ):
        return 'close_case'

    if state.get('role_top_share', 0.0) >= 0.80 and state['elapsed_hours'] > state['delay_q75']:
        if 'assign_to_primary_team' in valid_actions:
            return 'assign_to_primary_team'

    if state['branch_label'] == 'suspension' and 'escalate_to_higher_authority' in valid_actions:
        return 'escalate_to_higher_authority'
    if state['branch_label'] in {'refusal', 'suspension'} and 'trigger_high_cost_escalation' in valid_actions:
        return 'trigger_high_cost_escalation'
    if 'prioritize_urgent_case' in valid_actions and state['elapsed_hours'] > state['delay_q75']:
        return 'prioritize_urgent_case'
    if 'rebalance_overloaded_queue' in valid_actions and state['rework_count'] >= 2:
        return 'rebalance_overloaded_queue'
    if 'enable_cross_trained_pool' in valid_actions and state.get('role_candidate_count', 0) >= 6:
        return 'enable_cross_trained_pool'

    return DEFAULT_ACTION


def simulate_case_process(
    env, municipality, case_id, policy_fn,
    max_steps, base_start_timestamp, delay_q75, delay_q90,
    trace_rows, episode_rows, rng
):
    """Role-aware SimPy process: run one case trajectory and append trace + episode summaries."""
    current_activity = sample_start_activity(municipality, rng)
    previous_activity = None
    seen_counts = defaultdict(int)
    action_counts = Counter()

    expected_trace_len_raw = float(avg_trace_len_by_m.get(municipality, max_steps / 2))
    expected_trace_len = max(expected_trace_len_raw * TRACE_LEN_TARGET_MULT, 6.0)
    completion_rate_target = float(completion_rate_by_m.get(municipality, 0.85))
    target_completion = bool(rng.random() < completion_rate_target)

    start_env_time = float(env.now)
    terminal_reason = None
    case_completed = False
    step_index = 0

    case_row_indices = []
    role_top_share_sum = 0.0

    while step_index < max_steps:
        elapsed_hours = float(env.now - start_env_time)
        rework_count = int(seen_counts[current_activity])
        seen_counts[current_activity] += 1

        text = current_activity.lower()
        if any(k in text for k in ['refusal', 'objection']):
            branch_label = 'refusal'
            branch_confidence = 0.8
        elif any(k in text for k in ['suspend', 'suspension']):
            branch_label = 'suspension'
            branch_confidence = 0.8
        elif any(k in text for k in ['complete', 'completeness', 'missing', 'supplement']):
            branch_label = 'completeness'
            branch_confidence = 0.8
        else:
            branch_label = 'unknown'
            branch_confidence = 0.4

        role_profile = get_role_profile(municipality, current_activity)
        role_actor_candidates = int(role_profile.get('actor_candidates', 0)) if isinstance(role_profile, dict) else 0
        role_actor_top_share = float(role_profile.get('top_actor_share', 0.0)) if isinstance(role_profile, dict) else 0.0
        role_top_share_sum += role_actor_top_share

        sim_resource, sim_actor = sample_role_assignment(municipality, current_activity, rng)

        base_options = get_transition_options(municipality, current_activity)

        state = {
            'municipality': municipality,
            'activity': current_activity,
            'elapsed_hours': elapsed_hours,
            'rework_count': rework_count,
            'branch_label': branch_label,
            'branch_confidence': branch_confidence,
            'delay_q75': delay_q75,
            'delay_q90': delay_q90,
            'step_index': step_index,
            'expected_trace_len': expected_trace_len,
            'max_steps': max_steps,
            'target_completion': target_completion,
            'role_candidate_count': role_actor_candidates,
            'role_top_share': role_actor_top_share,
            'role_has_mapping': bool(role_actor_candidates > 0),
            'sim_actor': sim_actor,
            'sim_resource': sim_resource,
            'is_terminal': False,
        }

        valid_actions = valid_actions_from_state(state)
        action = policy_fn(state, valid_actions)
        if action not in valid_actions:
            action = DEFAULT_ACTION
        action_counts[action] += 1

        adjusted_options = apply_action_to_transition_options(base_options, action)
        adjusted_options = dampen_loop_transitions(adjusted_options, seen_counts, current_activity=current_activity)

        next_activity = sample_from_weighted_pairs(adjusted_options, rng) if adjusted_options else None

        progress_ratio = (step_index + 1) / max(expected_trace_len, 1.0)
        if next_activity is not None and terminal_reason is None:
            if progress_ratio >= 1.10:
                base_stop_prob = 0.60
            elif progress_ratio >= 0.95:
                base_stop_prob = 0.30
            elif progress_ratio >= 0.80:
                base_stop_prob = 0.12
            else:
                base_stop_prob = 0.0

            stop_prob = base_stop_prob * STOP_PUSH * EXTRA_STOP_PUSH
            if rework_count >= 2:
                stop_prob = max(stop_prob, 0.08)
            if rework_count >= 3:
                stop_prob = max(stop_prob, 0.22)
            if rework_count >= 4:
                stop_prob = max(stop_prob, REWORK_FORCE_STOP_PROB)

            stop_prob = stop_prob if target_completion else (stop_prob * 0.65)
            stop_prob = float(np.clip(stop_prob, 0.0, 0.98))

            if float(rng.random()) < stop_prob:
                next_activity = None
                terminal_reason = 'stochastic_stop'
                case_completed = bool(target_completion)

        if action == 'close_case' and target_completion:
            next_activity = None
            terminal_reason = 'action_close_case'
            case_completed = True

        if next_activity is None and terminal_reason is None:
            terminal_reason = 'no_outgoing_transition'
            completion_text_signal = any(k in text for k in ['close', 'archived', 'decision sent', 'irrevocable'])
            case_completed = bool(target_completion and completion_text_signal)

        duration_hours = sample_duration_hours(municipality, current_activity, rng)
        duration_hours = apply_action_to_duration(duration_hours, action)
        duration_hours = max(float(duration_hours), 1e-4)

        event_timestamp = base_start_timestamp + pd.to_timedelta(float(env.now), unit='h')
        event_id = f'{case_id}_E{step_index:04d}'

        trace_rows.append({
            'municipality': municipality,
            'case_id': case_id,
            'event_id': event_id,
            'timestamp': event_timestamp,
            'activity': current_activity,
            'prev_activity': previous_activity,
            'next_activity': next_activity,
            'step_index': step_index,
            'trace_length': None,
            'time_since_case_start_hours': elapsed_hours,
            'time_since_prev_hours': duration_hours if step_index > 0 else 0.0,
            'rework_count_activity': rework_count,
            'seen_activity_before': bool(rework_count > 0),
            'branch_label': branch_label,
            'branch_confidence': branch_confidence,
            'sim_resource': sim_resource,
            'sim_actor': sim_actor,
            'role_actor_candidates': role_actor_candidates,
            'role_actor_top_share': role_actor_top_share,
            'is_terminal_event': False,
            'case_completed': False,
            'target_completion': bool(target_completion),
            'sim_action': action,
            'sim_valid_actions': '|'.join(valid_actions),
        })
        case_row_indices.append(len(trace_rows) - 1)

        yield env.timeout(duration_hours)

        if next_activity is None:
            break

        previous_activity = current_activity
        current_activity = next_activity
        step_index += 1

    if terminal_reason is None:
        terminal_reason = 'max_steps_reached'
        case_completed = False

    end_env_time = float(env.now)
    case_duration_hours = max(end_env_time - start_env_time, 0.0)

    case_rows_count = len(case_row_indices)
    if case_rows_count > 0:
        for idx in case_row_indices:
            trace_rows[idx]['trace_length'] = case_rows_count
        last_idx = case_row_indices[-1]
        trace_rows[last_idx]['is_terminal_event'] = True
        trace_rows[last_idx]['case_completed'] = bool(case_completed)

    episode_record = {
        'municipality': municipality,
        'case_id': case_id,
        'steps': int(case_rows_count),
        'duration_hours': float(case_duration_hours),
        'loops': int(sum(1 for c in seen_counts.values() if c > 1)),
        'terminal_reason': terminal_reason,
        'completion_rate_target': float(completion_rate_target),
        'target_completion': bool(target_completion),
        'case_completed': bool(case_completed),
        'avg_role_top_share': float(role_top_share_sum / max(case_rows_count, 1)),
    }
    for action_name in ACTIONS:
        episode_record[f'action_{action_name}'] = int(action_counts.get(action_name, 0))

    episode_rows.append(episode_record)


print('Role mapping features loaded from file and integrated into simulation state/policy logic.')

In [ ]:
def simulate_case_process(
    env, municipality, case_id, policy_fn,
    max_steps, base_start_timestamp, delay_q75, delay_q90,
    trace_rows, episode_rows, rng, resource_pool
):
    """Role-aware SimPy process with data-driven duration/loop/stop calibration."""
    current_activity = sample_start_activity(municipality, rng)
    previous_activity = None
    seen_counts = defaultdict(int)
    action_counts = Counter()

    expected_trace_len = float(avg_trace_len_by_m.get(municipality, max_steps / 2))
    completion_rate_target = float(completion_rate_by_m.get(municipality, 0.85))
    target_completion = bool(rng.random() < completion_rate_target)

    start_env_time = float(env.now)
    terminal_reason = None
    case_completed = False
    step_index = 0

    case_row_indices = []
    role_top_share_sum = 0.0

    while step_index < max_steps:
        elapsed_hours = float(env.now - start_env_time)
        rework_count = int(seen_counts[current_activity])
        seen_counts[current_activity] += 1

        text = current_activity.lower()
        if any(k in text for k in ['refusal', 'objection']):
            branch_label = 'refusal'
            branch_confidence = 0.8
        elif any(k in text for k in ['suspend', 'suspension']):
            branch_label = 'suspension'
            branch_confidence = 0.8
        elif any(k in text for k in ['complete', 'completeness', 'missing', 'supplement']):
            branch_label = 'completeness'
            branch_confidence = 0.8
        else:
            branch_label = 'unknown'
            branch_confidence = 0.4

        role_profile = get_role_profile(municipality, current_activity)
        role_actor_candidates = int(role_profile.get('actor_candidates', 0)) if isinstance(role_profile, dict) else 0
        role_actor_top_share = float(role_profile.get('top_actor_share', 0.0)) if isinstance(role_profile, dict) else 0.0
        role_top_share_sum += role_actor_top_share

        sim_resource, sim_actor = sample_role_assignment(municipality, current_activity, rng)

        state = {
            'municipality': municipality,
            'activity': current_activity,
            'elapsed_hours': elapsed_hours,
            'rework_count': rework_count,
            'branch_label': branch_label,
            'branch_confidence': branch_confidence,
            'delay_q75': delay_q75,
            'delay_q90': delay_q90,
            'step_index': step_index,
            'expected_trace_len': expected_trace_len,
            'max_steps': max_steps,
            'target_completion': target_completion,
            'role_candidate_count': role_actor_candidates,
            'role_top_share': role_actor_top_share,
            'role_has_mapping': bool(role_actor_candidates > 0),
            'sim_actor': sim_actor,
            'sim_resource': sim_resource,
            'is_terminal': False,
        }

        valid_actions = valid_actions_from_state(state)
        action = policy_fn(state, valid_actions)
        if action not in valid_actions:
            action = DEFAULT_ACTION
        action_counts[action] += 1

        base_options = get_transition_options(municipality, current_activity)
        adjusted_options = apply_action_to_transition_options(base_options, action)
        adjusted_options = dampen_loop_transitions(
            adjusted_options,
            seen_counts,
            municipality=municipality,
            current_activity=current_activity,
        )

        next_activity = sample_from_weighted_pairs(adjusted_options, rng) if adjusted_options else None

        progress_ratio = (step_index + 1) / max(expected_trace_len, 1.0)
        if next_activity is not None and terminal_reason is None:
            stop_prob = stochastic_stop_probability(
                municipality=municipality,
                progress_ratio=progress_ratio,
                rework_count=rework_count,
            )
            stop_prob = float(np.clip(stop_prob, 0.0, 0.98))

            if float(rng.random()) < stop_prob:
                next_activity = None
                terminal_reason = 'stochastic_stop'
                completion_text_signal = any(k in text for k in ['close', 'archived', 'decision sent', 'irrevocable'])
                case_completed = bool(target_completion and (completion_text_signal or progress_ratio >= 1.0))

        if action == 'close_case' and target_completion:
            next_activity = None
            terminal_reason = 'action_close_case'
            case_completed = True

        if next_activity is None and terminal_reason is None:
            terminal_reason = 'no_outgoing_transition'
            completion_text_signal = any(k in text for k in ['close', 'archived', 'decision sent', 'irrevocable'])
            case_completed = bool(target_completion and completion_text_signal)

        sampled_duration = sample_duration_hours(municipality, current_activity, rng)
        sampled_duration = apply_action_to_duration(sampled_duration, action)
        sampled_duration = max(float(sampled_duration), 1e-4)

        queue_wait_hours = 0.0
        service_duration_hours = sampled_duration
        request_start = float(env.now)
        with resource_pool.request() as req:
            yield req
            queue_wait_hours = max(float(env.now) - request_start, 0.0)
            yield env.timeout(service_duration_hours)

        total_step_hours = queue_wait_hours + service_duration_hours

        event_timestamp = base_start_timestamp + pd.to_timedelta(float(env.now), unit='h')
        event_id = f'{case_id}_E{step_index:04d}'

        trace_rows.append({
            'municipality': municipality,
            'case_id': case_id,
            'event_id': event_id,
            'timestamp': event_timestamp,
            'activity': current_activity,
            'prev_activity': previous_activity,
            'next_activity': next_activity,
            'step_index': step_index,
            'trace_length': None,
            'time_since_case_start_hours': float(env.now - start_env_time),
            'time_since_prev_hours': total_step_hours if step_index > 0 else 0.0,
            'queue_wait_hours': queue_wait_hours,
            'service_duration_hours': service_duration_hours,
            'resource_capacity': int(resource_pool.capacity),
            'resource_queue_at_request': int(resource_pool.count),
            'rework_count_activity': rework_count,
            'seen_activity_before': bool(rework_count > 0),
            'branch_label': branch_label,
            'branch_confidence': branch_confidence,
            'sim_resource': sim_resource,
            'sim_actor': sim_actor,
            'role_actor_candidates': role_actor_candidates,
            'role_actor_top_share': role_actor_top_share,
            'is_terminal_event': False,
            'case_completed': False,
            'target_completion': bool(target_completion),
            'sim_action': action,
            'sim_valid_actions': '|'.join(valid_actions),
        })
        case_row_indices.append(len(trace_rows) - 1)

        if next_activity is None:
            break

        previous_activity = current_activity
        current_activity = next_activity
        step_index += 1

    if terminal_reason is None:
        terminal_reason = 'max_steps_reached'
        case_completed = False

    end_env_time = float(env.now)
    case_duration_hours = max(end_env_time - start_env_time, 0.0)

    case_rows_count = len(case_row_indices)
    if case_rows_count > 0:
        for idx in case_row_indices:
            trace_rows[idx]['trace_length'] = case_rows_count
        last_idx = case_row_indices[-1]
        trace_rows[last_idx]['is_terminal_event'] = True
        trace_rows[last_idx]['case_completed'] = bool(case_completed)

    episode_record = {
        'municipality': municipality,
        'case_id': case_id,
        'steps': int(case_rows_count),
        'duration_hours': float(case_duration_hours),
        'loops': int(sum(1 for c in seen_counts.values() if c > 1)),
        'terminal_reason': terminal_reason,
        'completion_rate_target': float(completion_rate_target),
        'target_completion': bool(target_completion),
        'case_completed': bool(case_completed),
        'avg_role_top_share': float(role_top_share_sum / max(case_rows_count, 1)),
    }
    for action_name in ACTIONS:
        episode_record[f'action_{action_name}'] = int(action_counts.get(action_name, 0))

    episode_rows.append(episode_record)


print('Data-driven simulation mode active: fitted duration, loop, and stop models enabled.')

In [ ]:
# Simulation configuration
MUNICIPALITIES = sorted(set(int(x) for x in features_df['municipality'].dropna().unique()))
N_CASES_PER_MUNICIPALITY = 120
N_SIM_RUNS = 20
BASE_SEED = 88

MAX_STEPS = int(features_df.groupby(['municipality', 'case_id'])['step_index'].max().quantile(0.90) + 1)
MAX_STEPS = max(MAX_STEPS, 25)

delay_q75 = float(features_df['time_since_case_start_hours'].quantile(0.75))
delay_q90 = float(features_df['time_since_case_start_hours'].quantile(0.90))

base_start_timestamp = pd.Timestamp('2011-01-01T00:00:00Z')

all_trace_rows = []
all_episode_rows = []
run_summary_rows = []

for run_id in range(1, N_SIM_RUNS + 1):
    run_seed = BASE_SEED + run_id - 1
    run_rng = np.random.default_rng(run_seed)

    run_trace_rows = []
    run_episode_rows = []

    for m in MUNICIPALITIES:
        env = simpy.Environment()
        mean_interarrival = arrival_mean_hours_by_m.get(m, 24.0)

        capacity = int(resource_capacity_by_m.get(m, default_resource_capacity))
        capacity = max(capacity, 1)
        resource_pool = simpy.Resource(env, capacity=capacity)

        env.process(case_arrival_generator(
            env=env,
            municipality=m,
            n_cases=N_CASES_PER_MUNICIPALITY,
            mean_interarrival_hours=mean_interarrival,
            policy_fn=heuristic_policy,
            max_steps=MAX_STEPS,
            base_start_timestamp=base_start_timestamp,
            delay_q75=delay_q75,
            delay_q90=delay_q90,
            trace_rows=run_trace_rows,
            episode_rows=run_episode_rows,
            rng=run_rng,
            resource_pool=resource_pool,
            run_id=run_id,
        ))

        env.run()

    run_trace_df = pd.DataFrame(run_trace_rows)
    run_episode_df = pd.DataFrame(run_episode_rows)

    if not run_trace_df.empty:
        run_trace_df['run_id'] = int(run_id)
    if not run_episode_df.empty:
        run_episode_df['run_id'] = int(run_id)

    all_trace_rows.extend(run_trace_rows)
    all_episode_rows.extend(run_episode_rows)

    run_summary_rows.append({
        'run_id': int(run_id),
        'seed': int(run_seed),
        'trace_rows': int(len(run_trace_df)),
        'episodes': int(len(run_episode_df)),
        'mean_steps': float(run_episode_df['steps'].mean()) if len(run_episode_df) else np.nan,
        'mean_duration_hours': float(run_episode_df['duration_hours'].mean()) if len(run_episode_df) else np.nan,
        'completion_rate': float(run_episode_df['case_completed'].mean()) if len(run_episode_df) else np.nan,
        'mean_loops': float(run_episode_df['loops'].mean()) if len(run_episode_df) else np.nan,
        'mean_queue_wait_hours': float(run_trace_df['queue_wait_hours'].mean()) if 'queue_wait_hours' in run_trace_df.columns and len(run_trace_df) else np.nan,
    })

    print(f'Run {run_id:02d}/{N_SIM_RUNS}: seed={run_seed}, trace_rows={len(run_trace_df):,}, episodes={len(run_episode_df):,}')

sim_trace_df = pd.DataFrame(all_trace_rows)
sim_episode_df = pd.DataFrame(all_episode_rows)

if not sim_trace_df.empty:
    sim_trace_df['run_id'] = sim_trace_df['case_id'].astype(str).str.extract(r'^R(\d+)_').astype(float).astype('Int64')
if not sim_episode_df.empty:
    sim_episode_df['run_id'] = sim_episode_df['case_id'].astype(str).str.extract(r'^R(\d+)_').astype(float).astype('Int64')

sim_trace_df = sim_trace_df.sort_values(['run_id', 'municipality', 'case_id', 'step_index']).reset_index(drop=True)
sim_episode_df = sim_episode_df.sort_values(['run_id', 'municipality', 'case_id']).reset_index(drop=True)
sim_run_summary_df = pd.DataFrame(run_summary_rows).sort_values('run_id').reset_index(drop=True)

sim_trace_df.to_csv(SIM_TRACE_PATH, index=False)
sim_episode_df.to_csv(SIM_EPISODE_PATH, index=False)
sim_run_summary_df.to_csv(SIM_RUN_SUMMARY_PATH, index=False)

print('Saved trace table   :', SIM_TRACE_PATH.resolve())
print('Saved episode table :', SIM_EPISODE_PATH.resolve())
print('Saved run summary   :', SIM_RUN_SUMMARY_PATH.resolve())
print('MAX_STEPS used:', MAX_STEPS)
print('Total trace rows:', len(sim_trace_df))
print('Total episodes  :', len(sim_episode_df))
print('Unique runs     :', sim_trace_df['run_id'].nunique() if 'run_id' in sim_trace_df.columns else 1)
sim_trace_df.head(5)

In [ ]:
# Step 6 checks

# Check 1: no impossible transitions outside allowed transition table
transition_rows = sim_trace_df[sim_trace_df['next_activity'].notna()].copy()
invalid_transition_rows = []
for _, row in transition_rows.iterrows():
    m = int(row['municipality'])
    edge = (str(row['activity']), str(row['next_activity']))
    if edge not in allowed_edges_by_m.get(m, set()):
        invalid_transition_rows.append({
            'municipality': m,
            'case_id': row['case_id'],
            'activity': row['activity'],
            'next_activity': row['next_activity'],
        })

invalid_transition_df = pd.DataFrame(invalid_transition_rows)
invalid_transition_count = int(len(invalid_transition_df))

# Check 2: all episodes terminate within max step cap
over_cap_count = int((sim_episode_df['steps'] > MAX_STEPS).sum())

# Check 3: completion behavior should not collapse to all-true or all-false
sim_completion_rate = float(sim_episode_df['case_completed'].mean())
real_completion_rate = float(
    features_df.groupby(['municipality', 'case_id'])['case_completed'].max().astype(float).mean()
)
completion_not_degenerate = 0.01 < sim_completion_rate < 0.99

print('Invalid transition rows:', invalid_transition_count)
print('Episodes over max step cap:', over_cap_count)
print('Sim completion rate:', round(sim_completion_rate, 4))
print('Real completion rate:', round(real_completion_rate, 4))
print('Terminal reason mix:')
print(sim_episode_df['terminal_reason'].value_counts(normalize=True).round(4).head(10))

print('Check A passed:', invalid_transition_count == 0)
print('Check B passed:', over_cap_count == 0)
print('Check C passed:', completion_not_degenerate)

if invalid_transition_count > 0:
    display(invalid_transition_df.head(20))

In [ ]:
# ── Multi-run diagnostics dashboard (all 20 runs) ─────────────────────────
import matplotlib.ticker as mticker

if 'run_id' not in sim_episode_df.columns:
    sim_episode_df['run_id'] = 1
if 'run_id' not in sim_trace_df.columns:
    sim_trace_df['run_id'] = 1

sim_episode_df['run_id'] = pd.to_numeric(sim_episode_df['run_id'], errors='coerce').fillna(1).astype(int)
sim_trace_df['run_id'] = pd.to_numeric(sim_trace_df['run_id'], errors='coerce').fillna(1).astype(int)

run_m_summary = (
    sim_episode_df.groupby(['run_id', 'municipality'])
    .agg(
        mean_steps=('steps', 'mean'),
        mean_duration_hours=('duration_hours', 'mean'),
        completion_rate=('case_completed', 'mean'),
        mean_loops=('loops', 'mean'),
        n_cases=('case_id', 'size'),
    )
    .reset_index()
 )

run_summary_df = (
    sim_episode_df.groupby('run_id')
    .agg(
        episodes=('case_id', 'size'),
        mean_steps=('steps', 'mean'),
        mean_duration_hours=('duration_hours', 'mean'),
        completion_rate=('case_completed', 'mean'),
        mean_loops=('loops', 'mean'),
    )
    .reset_index()
    .sort_values('run_id')
)

# Run-level quality checks
transition_rows = sim_trace_df[sim_trace_df['next_activity'].notna()].copy()
invalid_by_run = {}
for run_id, grp in transition_rows.groupby('run_id'):
    invalid = 0
    for _, row in grp.iterrows():
        m = int(row['municipality'])
        edge = (str(row['activity']), str(row['next_activity']))
        if edge not in allowed_edges_by_m.get(m, set()):
            invalid += 1
    invalid_by_run[int(run_id)] = int(invalid)

overcap_by_run = (
    sim_episode_df.assign(over_cap=sim_episode_df['steps'] > MAX_STEPS)
    .groupby('run_id')['over_cap']
    .sum()
    .astype(int)
    .to_dict()
)

run_summary_df['invalid_transitions'] = run_summary_df['run_id'].map(invalid_by_run).fillna(0).astype(int)
run_summary_df['episodes_over_cap'] = run_summary_df['run_id'].map(overcap_by_run).fillna(0).astype(int)

real_completion_rate = float(
    features_df.groupby(['municipality', 'case_id'])['case_completed'].max().astype(float).mean()
)

run_ids = sorted(run_summary_df['run_id'].unique())
municipalities = sorted(run_m_summary['municipality'].unique())
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(2, 3, figsize=(19, 10))
fig.suptitle(f'Step 6 simulator diagnostics across {len(run_ids)} runs', fontsize=14, fontweight='bold', y=1.02)

# Panel 1: mean steps per run (per municipality + global)
ax = axes[0, 0]
for i, m in enumerate(municipalities):
    sub = run_m_summary[run_m_summary['municipality'] == m].sort_values('run_id')
    ax.plot(sub['run_id'], sub['mean_steps'], marker='o', ms=3.5, color=colors[i % len(colors)], alpha=0.8, label=f'M{m}')
ax.plot(run_summary_df['run_id'], run_summary_df['mean_steps'], color='black', linewidth=2.0, label='Global mean')
ax.set_title('Mean steps per case by run')
ax.set_xlabel('Run ID')
ax.set_ylabel('Mean steps')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, linestyle='--', alpha=0.35)
ax.legend(fontsize=7, ncol=2)

# Panel 2: completion rate per run vs real baseline
ax = axes[0, 1]
for i, m in enumerate(municipalities):
    sub = run_m_summary[run_m_summary['municipality'] == m].sort_values('run_id')
    ax.plot(sub['run_id'], sub['completion_rate'], marker='^', ms=3.5, color=colors[i % len(colors)], alpha=0.8, label=f'M{m}')
ax.plot(run_summary_df['run_id'], run_summary_df['completion_rate'], color='black', linewidth=2.0, label='Global mean')
ax.axhline(real_completion_rate, color='red', linestyle='--', linewidth=1.5, label='Real baseline')
ax.set_title('Completion rate by run')
ax.set_xlabel('Run ID')
ax.set_ylabel('Completed fraction')
ax.set_ylim(-0.02, 1.02)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, linestyle='--', alpha=0.35)
ax.legend(fontsize=7, ncol=2)

# Panel 3: mean duration (days) per run
ax = axes[0, 2]
for i, m in enumerate(municipalities):
    sub = run_m_summary[run_m_summary['municipality'] == m].sort_values('run_id')
    ax.plot(sub['run_id'], sub['mean_duration_hours'] / 24.0, marker='s', ms=3.5, color=colors[i % len(colors)], alpha=0.8, label=f'M{m}')
ax.plot(run_summary_df['run_id'], run_summary_df['mean_duration_hours'] / 24.0, color='black', linewidth=2.0, label='Global mean')
ax.set_title('Mean case duration by run')
ax.set_xlabel('Run ID')
ax.set_ylabel('Duration (days)')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, linestyle='--', alpha=0.35)
ax.legend(fontsize=7, ncol=2)

# Panel 4: mean loops by run
ax = axes[1, 0]
for i, m in enumerate(municipalities):
    sub = run_m_summary[run_m_summary['municipality'] == m].sort_values('run_id')
    ax.plot(sub['run_id'], sub['mean_loops'], marker='D', ms=3.5, color=colors[i % len(colors)], alpha=0.8, label=f'M{m}')
ax.plot(run_summary_df['run_id'], run_summary_df['mean_loops'], color='black', linewidth=2.0, label='Global mean')
ax.set_title('Mean loops per case by run')
ax.set_xlabel('Run ID')
ax.set_ylabel('Mean loops / case')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, linestyle='--', alpha=0.35)
ax.legend(fontsize=7, ncol=2)

# Panel 5: terminal reason composition by run (stacked)
ax = axes[1, 1]
if 'terminal_reason' in sim_episode_df.columns:
    term = (
        sim_episode_df.groupby(['run_id', 'terminal_reason'])
        .size()
        .reset_index(name='count')
    )
    total = term.groupby('run_id')['count'].transform('sum')
    term['fraction'] = term['count'] / total

    top_reasons = (
        term.groupby('terminal_reason')['count'].sum().sort_values(ascending=False).head(5).index.tolist()
    )
    term['terminal_reason_plot'] = np.where(term['terminal_reason'].isin(top_reasons), term['terminal_reason'], 'other')
    term_plot = (
        term.groupby(['run_id', 'terminal_reason_plot'])['fraction']
        .sum()
        .unstack(fill_value=0.0)
        .sort_index()
    )

    bottom = np.zeros(len(term_plot))
    for i, reason in enumerate(term_plot.columns):
        vals = term_plot[reason].values
        ax.bar(term_plot.index, vals, bottom=bottom, label=reason, alpha=0.85)
        bottom += vals

    ax.set_ylim(0, 1.0)
    ax.set_title('Terminal reason mix by run')
    ax.set_xlabel('Run ID')
    ax.set_ylabel('Fraction of episodes')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(fontsize=7, ncol=2)
else:
    ax.set_visible(False)

# Panel 6: run quality checks
ax = axes[1, 2]
x = run_summary_df['run_id'].to_numpy(dtype=float)
invalid_vals = run_summary_df['invalid_transitions'].to_numpy(dtype=float)
overcap_vals = run_summary_df['episodes_over_cap'].to_numpy(dtype=float)
w = 0.40
ax.bar(x - w/2, invalid_vals, width=w, label='Invalid transitions')
ax.bar(x + w/2, overcap_vals, width=w, label='Episodes > MAX_STEPS')
all_zero_quality = (len(x) > 0) and np.all(invalid_vals == 0) and np.all(overcap_vals == 0)
if all_zero_quality:
    ax.set_ylim(0, 1)
    ax.text(0.5, 0.5, 'All quality checks are zero', transform=ax.transAxes, ha='center', va='center', fontsize=9, color='dimgray')
ax.set_title('Quality checks by run')
ax.set_xlabel('Run ID')
ax.set_ylabel('Count')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(True, axis='y', linestyle='--', alpha=0.35)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('\nRun-level summary (mean ± std across runs):')
summary_stats = run_summary_df[['mean_steps', 'mean_duration_hours', 'completion_rate', 'mean_loops']].agg(['mean', 'std']).T
display(summary_stats)
display(run_summary_df.head(20))

## Step 6 complete

This notebook saves:
- `output/sim_trace_table.csv`
- `output/sim_episode_summary.csv`
- `output/sim_run_summary.csv`

Next: compare simulated behavior vs real logs in Step 7.